# Solutions · Chapter 02-02 · Where data comes from

Worked answers with reasoning. E7 and E8 are a pair worth doing together: the correction works, and
then you find out what it quietly assumed and how much it left behind.

Self-contained: run from the top with a fresh kernel.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(12)
n_days = 120
day = pd.date_range("2024-01-01", periods=n_days, freq="D")
true_demand = rng.poisson(100, n_days)
true_night = rng.poisson(2, n_days)
upgraded = np.arange(n_days) >= 60
staff_moves = np.where(upgraded, rng.poisson(25, n_days), 0)
staff_at_night = np.where(upgraded, rng.poisson(20, n_days), 0)

recorded = pd.DataFrame({"day": day, "rentals": true_demand + staff_moves,
                         "night_rentals": true_night + staff_at_night}).set_index("day")
truth = pd.Series(true_demand, index=day, name="true_demand")   # only we can see this
print("recorded mean before/after:", round(recorded['rentals'][:60].mean(), 1),
      "/", round(recorded['rentals'][60:].mean(), 1))

## E1 · The six questions

1. Who or what created this record, and by what mechanism?
2. Why was it collected - what purpose was it serving?
3. What exactly does each field mean, in the collector's words?
4. When was it recorded, relative to the event it describes?
5. What has changed over time - systems, definitions, thresholds, vendors?
6. Who or what is missing entirely?

**The March surge turned on question 5.** The mechanism, the purpose and the field meanings were all
stable; what changed was the firmware, and with it the definition of an undocking event. Question 5
is the one that produces a *plausible* wrong answer rather than an obviously broken one, which is
why it is the hardest to catch and the most expensive to miss.

## E2 · Why the "cannot have changed" slice works

A measurement change usually affects **everything the instrument records**. A real change usually
affects only **the part of the world that actually changed**. So if you can find a slice where the
real quantity is pinned - by physics, by opening hours, by law, by a group the change could not
reach - then any movement in that slice must come from the instrument.

It works because it converts a question about the world, which you cannot check, into a question
about a subset, which you can.

**Examples from other domains:**

- **Website analytics.** Traffic jumps 30%. Check traffic from a country you do not operate in, or
  requests to a URL that no longer exists. If those jumped too, a tracking script was double-firing.
- **Clinical measurement.** A hospital's average blood pressure readings shift. Check the readings
  taken on the calibration phantom or on staff volunteers - people whose blood pressure did not
  change as a group. A shift there is the machine.
- **Sales.** Revenue rises after a pricing change. Check a product line whose price did not change,
  in a region the campaign did not reach. That is a control group, and it is the same idea as the
  randomised comparison in 00-04, found rather than designed.
- **Sensors generally.** Any reading during a period when the thing being measured was switched
  off. Non-zero output from an idle instrument is pure measurement.

**The general form:** *what part of this data should not have moved?* If you cannot name one, you
cannot distinguish measurement from world, and you should say so rather than pick.

## E3 · Billing data, reliable and not

**Reliable:** fields that someone would dispute or audit. For an online shop - the amount charged,
the payment method, the delivery address, the tax rate, the order timestamp. Errors in these produce
complaints, refunds and regulatory attention, so they are checked by people whose job it is.

**Unreliable:** fields that no money depends on. The reason-for-return dropdown, the "how did you
hear about us" field, the product category on a legacy SKU, the customer's job title. Nobody
downstream notices when they are wrong.

**The asymmetry is systematic and worth predicting in advance.** Before opening a dataset, ask which
fields are *load-bearing* for the system that produced it. Those are trustworthy. Everything else is
best-effort - and it is very often exactly the field a model finds most predictive, because a field
that is filled in inconsistently encodes *who* filled it in and *when they had time*.

**A concrete trap:** "reason for return" is left blank when the warehouse is busy. Busy weeks are
high-volume weeks. A model using that field learns something real about warehouse staffing and
nothing about returns.

## E4 · How much does the night explain?

In [ ]:
before = recorded.loc[:"2024-02-29"]
after = recorded.loc["2024-03-01":]

night_excess = after["night_rentals"].mean() - before["night_rentals"].mean()
total_excess = after["rentals"].mean() - before["rentals"].mean()

print(f"night share before : {before['night_rentals'].mean() / before['rentals'].mean():.2%}")
print(f"night share after  : {after['night_rentals'].mean() / after['rentals'].mean():.2%}")
print(f"night excess       : {night_excess:.1f} events per day")
print(f"total excess       : {total_excess:.1f} events per day")
print(f"the night explains : {night_excess / total_excess:.0%} of the increase")

- Night share: **1.85%** before, **16.54%** after.
- Night excess: **19.0** events per day. Total excess: **27.0** per day.
- **The night explains about 70% of the increase.**

The remaining 30% - roughly 8 events a day - are staff movements happening during opening hours,
where they are indistinguishable from customers.

**That residual is the important part of this exercise.** The night slice proved the contamination
exists and measured most of it. It did not measure all of it, and nothing in the data tells you how
much is left. Any correction built on the night counts alone will under-correct by an unknown
amount - which is E7 and E8.

**The general lesson:** a diagnostic slice is excellent for *detection* and unreliable for
*correction*. Detecting that a measurement changed is usually easy once you look. Recovering what the
measurement would have been is usually impossible without help from whoever changed it.

## E5 · Three sentences

**If the growth were real:**

> "Daily rentals rose 27% in March and held through April, driven by demand rather than by any
> change in how we count. Night-time undockings stayed flat, so the growth is genuine customer
> activity."

**As it should be written, given what we know:**

> "Recorded undockings rose 27% on 1 March, the day the dock firmware was updated. Night-time
> undockings rose more than tenfold on the same date, which customer demand cannot explain. We
> believe the counter now includes staff bike movements; the series is not comparable across 1 March
> and we are asking the vendor to confirm."

**As it would be written by someone who wanted the bonus:**

> "March was our best month ever, up 27% on February."

Every word of the third is true. It reports a real measured increase over a real prior month. It is
the same technique as the chosen window in 01-06 and the "best of five seeds" in 01-06's E10:
nothing false is stated, and the reader draws a conclusion the evidence does not support.

**What makes the second version professional is not caution, it is that it names the alternative
explanation and says what would settle it.** A sentence that cannot be checked is an opinion.

## E6 · Finding the step automatically

In [ ]:
def find_step(series, min_side=10):
    """Scan every split point; return the date where the mean jumps most, in relative terms."""
    best = (0.0, None, None, None)
    for i in range(min_side, len(series) - min_side):
        left, right = series.iloc[:i].mean(), series.iloc[i:].mean()
        jump = abs(right - left) / max(left, 1e-9)
        if jump > best[0]:
            best = (jump, series.index[i], left, right)
    return best


for column in ("rentals", "night_rentals"):
    jump, when, left, right = find_step(recorded[column])
    print(f"{column:<14} largest step at {when.date()}  {left:.1f} -> {right:.1f}  ({jump:+.0%})")

**Both columns point at the same date, 29 February** - one day before the firmware actually changed.
They agree with each other exactly, and that agreement is the finding.

Two independent series changing on the same day is far stronger evidence than either alone. A demand
surge would move the total and leave the night alone; a definition change moves both.

The one-day offset is worth noticing rather than ignoring. The scan compares means either side of
each candidate split and picks the largest jump; with daily noise, the best split can land a day
either side of the true break. **A changepoint scan locates a date to ask about, to within a day or
two - it does not identify the date.** Reporting "the change is around the end of February" and then
checking the deployment log is right; reporting "the change was on 29 February" is over-claiming from
a noisy estimate.

**Three honest limitations of this function**, which matter if you are tempted to trust it:

- **It always returns an answer.** Run it on pure noise and it will confidently name a date. It finds
  the largest step, not a *significant* one, and there is always a largest.
- **It assumes exactly one step.** Two changes, or a gradual drift, will produce a misleading single
  answer.
- **It has no notion of uncertainty.** The proper version tests whether the jump is larger than
  ordinary variation - which needs 03-03.

Use it as a *pointer to a date to ask about*, never as evidence on its own. The evidence is the
changelog.

## E7 · Correcting the series

In [ ]:
night_baseline = before["night_rentals"].mean()
contamination = (recorded["night_rentals"] - night_baseline).clip(lower=0)
corrected = (recorded["rentals"] - contamination).rename("corrected")

comparison = pd.DataFrame({"recorded": recorded["rentals"], "corrected": corrected, "truth": truth})
print(comparison.loc["2024-03-01":].mean().round(1).to_string())
print(f"\nremaining bias after correction: "
      f"{comparison.loc['2024-03-01':, 'corrected'].mean() - comparison.loc['2024-03-01':, 'truth'].mean():+.1f} per day")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(comparison.index, comparison["recorded"], color="#D55E00", linewidth=0.8, label="recorded")
ax.plot(comparison.index, comparison["corrected"], color="#0072B2", linewidth=0.8, label="corrected")
ax.plot(comparison.index, comparison["truth"], color="#009E73", linewidth=1.4, alpha=0.7, label="true demand")
ax.axvline(day[60], color="grey", linestyle="--", linewidth=1)
ax.set_xlabel("Date"); ax.set_ylabel("Rentals per day")
ax.set_title("The correction removes most of the contamination, not all of it")
ax.legend(fontsize=8)
plt.show()

The correction pulls the post-March mean from **126.0** down to **107.0**, against a true demand of
**101.2**. It removes about three quarters of the error and leaves **5.8** rentals a day.

**The assumption it rests on, stated plainly:**

> *All of the contamination is visible in the night window.*

Equivalently: staff never move bikes during opening hours. We know from the generator that they do -
about 5 of the 25 daily staff moves happen in daylight - so the correction under-removes by exactly
that amount, and nothing in the data reveals it.

**Why this is still worth doing.** A series that is 7% wrong is much better than one that is 27%
wrong, *provided you say which*. The failure mode is presenting the corrected series as if it were
the truth. The honest output is a corrected series plus a stated residual uncertainty: "we believe
this removes most but not all of the staff contamination; the remainder is likely 5-10 per day."

## E8 · What the correction assumes, and how it fails

**The assumption:** the night window captures all of the contamination - staff move bikes only when
the stand is closed.

**A situation where it fails:** a rebalancing van that runs at 7am and 6pm, inside opening hours.
Then the night window sees a fraction of the staff moves, the correction removes that fraction, and
the residual bias is large and invisible. In the limit - staff who work only during the day - the
night window shows nothing at all and the whole method reports "no contamination" while the totals
are 25% inflated.

**Evidence that would let you check it:**

- **The staff rota or the van's GPS log.** Direct, decisive, and someone has it.
- **The hourly profile before and after.** If contamination were night-only, the two profiles would
  differ only at night. A change in the 7am and 6pm bars says otherwise, and it needs no help from
  anyone.
- **Undocking duration.** A staff move re-docks within a minute or two; a customer rental does not.
  If the raw event log has durations, the distribution of very short trips before and after the
  update would identify staff moves directly - and would give a much better correction than the
  night proxy.
- **Dock-to-dock pairs.** Staff moves go from full racks to empty ones, systematically. Customer
  trips do not.

**The lesson worth carrying:** every correction is a model with assumptions, and it deserves the same
scepticism as the thing it corrects. Write the assumption down next to the corrected number. A
correction whose assumption is unstated is more dangerous than no correction, because it looks like
a fix.

## E9 · Accuracy drops for customers who joined after a certain month

Four candidates, cheapest to check first:

1. **A field changed meaning or stopped being populated for new customers.** A signup form was
   redesigned; a field that used to be mandatory became optional; a default changed. **Check:** the
   fill rate and value distribution of every feature, by signup month. A column that goes 98%
   populated to 40% at exactly the boundary is the whole answer, and it is one `groupby`.
2. **A new acquisition channel.** Customers arriving from a different source genuinely behave
   differently, so the model - trained on the old mix - is being asked about a population it never
   saw. **Check:** the distribution of channel, region, plan and device by signup month. This is drift
   in the inputs, and it is 13-08.
3. **The label is defined differently, or is not yet observable.** If churn means "no activity for 90
   days", customers who joined recently cannot have churned yet, so their labels are systematically
   different - or wrong. **Check:** how the label is computed and how much history each cohort has.
4. **A pipeline or vendor change on the boundary date.** A new identity provider, a changed
   timestamp convention, a different upstream join. **Check:** the changelog - which means asking.

**What I would ask first:** *"what changed for customers who signed up after that month?"* - put to
whoever owns signup. It costs one message and usually returns the answer before any analysis
finishes. If nobody knows, question 1's fill-rate check is ten lines and settles most cases.

**Note that only one of the four is about the model.** That is the normal ratio, and it is why this
module comes before any modelling chapter.

## E10 · "Why spend the first day talking to people?"

> Because the file cannot tell me what its numbers mean, and getting that wrong invalidates
> everything after it. On this project the recorded rentals jumped 27% on 1 March, which looks like
> a demand surge and is a firmware update that started counting staff bike movements - a fact that
> exists in one engineer's head and in no column of the dataset. Had I gone straight to modelling, I
> would have trained a forecast on a level shift that will never repeat, over-ordered bikes for the
> rest of the year, and credited the jump to whatever marketing did that month. Thirty seconds of
> conversation replaces a week of forensics, and it also tells me which fields anyone actually checks
> - which predicts where the data is trustworthy before I look at a single value.

**What is being assessed:** whether you treat the dataset as evidence about a process rather than as
the ground truth. Candidates who answer "to understand the business" are saying something true and
vague; naming a specific failure that talking would have prevented is the difference.

## E11 · Five years of hospital admissions

| Question | Who to ask | Why it matters |
|---|---|---|
| What system produced these rows, and did it change in five years? | The clinical informatics or EHR team | Five years spans at least one system migration or coding-standard change |
| What exactly is a "readmission" here - which window, which hospital, which exclusions? | The clinicians and the quality team, together | Every hospital defines it differently; planned readmissions are usually excluded, and whether transfers count changes the rate substantially |
| When is each field populated relative to discharge? | Whoever runs the discharge workflow | Fields completed *after* the readmission is known are pure leakage, and coding is often finalised weeks later |
| Which patients are missing - transferred out, died, treated privately, no follow-up? | The registry owner | The outcome is unobservable for some patients, and they are not a random group |
| Did coding practice or reimbursement rules change? | Coding and billing | Diagnosis codes respond to reimbursement, so a "rise in a condition" can be a billing change |

**The answer that would most change my approach: the timing of field population.** If diagnosis codes
and discharge summaries are finalised days or weeks after discharge - which is normal - then most of
the richest features do not exist at the moment a readmission-risk prediction is needed. That does
not merely remove features; it changes what the model can be *for*. A model that runs at discharge
and one that runs at coding-completion are different products with different features and different
users, and discovering that in week one saves rebuilding in month three. It is the prediction-time
question from 00-01, and in healthcare it is decisive.

## E12 · Satisfaction from 3.8 to 4.4 in a week

**Three measurement explanations, before any explanation about customers:**

1. **Who was asked changed.** The survey trigger moved - now sent only after a completed delivery
   rather than after every order, or only to app users, or the reminder email stopped going to people
   who ignored the first one. **Slice to check:** response rate, and the mix of order type, channel
   and region among respondents, week by week. A score that moves because the *respondents* changed
   is 02-03's subject and is the most common cause by a wide margin.
2. **The instrument changed.** The scale was relabelled, a neutral option was removed, the question
   was reworded, the widget changed from stars to faces, or the default moved. **Slice to check:** the
   distribution of individual scores, not the mean. A rounding or scale change shows as a shift in
   *shape* - a spike at a new value, a missing category - which a mean hides completely.
3. **The aggregation changed.** A filter for suspected spam responses was added or removed, a
   duplicate-suppression rule changed, or "no response" started being excluded rather than counted.
   **Slice to check:** the response count and the number of unique respondents alongside the mean. A
   mean that rises while the count falls by a third is an aggregation change.

**Only after all three** would I consider that customers got happier - and even then, a real change
of 0.6 in one week for a stable business is implausibly fast. **Large, fast movements in a slow
metric are nearly always measurement.** Small, slow movements are where real change hides.

## E13 · Explaining it to Maria

> The counter went up by a quarter overnight on the first of March. On the same night, undockings
> between two and five in the morning went from about two to about twenty - and nobody takes a bike
> out at three in the morning. So something changed in what the counter counts, not in how many
> people are renting. I want to ask the dock company what they changed that day.

(69 words.)

**Why the 3am detail carries the whole argument:** Maria does not need to follow any reasoning about
measurement. She knows her own park at three in the morning, and she knows twenty people are not
renting bikes there. The argument lands because it appeals to something she already knows better
than you do - which is what a good explanation to a domain expert always does.

## E14 · What the mistake costs, in bikes

In [ ]:
from sklearn.linear_model import LinearRegression

t = np.arange(n_days).reshape(-1, 1)
future_t = np.arange(n_days, n_days + 30).reshape(-1, 1)

naive_forecast = LinearRegression().fit(t, recorded["rentals"]).predict(future_t)
corrected_forecast = LinearRegression().fit(t, corrected).predict(future_t)
true_level = true_demand.mean()

print(f"true demand level                    : {true_level:.1f} per day")
print(f"forecast from the recorded series    : {naive_forecast.mean():.1f} per day "
      f"({naive_forecast.mean() - true_level:+.1f})")
print(f"forecast from the corrected series   : {corrected_forecast.mean():.1f} per day "
      f"({corrected_forecast.mean() - true_level:+.1f})")
print(f"\nover 30 days, the uncorrected forecast over-orders "
      f"{(naive_forecast - true_level).sum():,.0f} bike-days")

**The uncorrected forecast runs 38 rentals a day too high, and over thirty days that is about 1,150
bike-days of demand that does not exist.** The corrected one is 11.5 too high.

Two features of that number are worth separating.

**First, it is not just the 27% step.** A trend model fitted across the discontinuity reads the jump
as *growth* and extrapolates it, so the forecast keeps climbing past the level the contaminated
series ever reached. A single step in the history becomes a rising line in the future - which is
worse than the step itself, and gets worse the further out you forecast.

**Second, the corrected forecast is still high** - by 11.5 a day, more than the 5.8 of level bias
E7 left behind, because the trend model extrapolates the residual step as well. The
correction improved the decision substantially and did not make it right.

**How Maria experiences this:** she hires a van and staff for a demand level that never arrives,
every week, until someone questions the forecast. The model will not flag it - its held-out error on
the contaminated history is *fine*, because the test period is contaminated in the same way. Only
reality disagrees, and reality reports slowly.

**The general point, which is the reason module 02 comes before module 05:** no amount of
cross-validation catches this. The evaluation and the training data share the same defect, so the
model looks correct by every internal measure. The only defence is having asked, on day one, what
changed and when.

---

## Where to go next

Back to the chapter for the mastery check and flashcards, then **02-03 · Sampling and selection
bias: the rows you never see**.